# Notebook 02 — YOLOv8 Detection Training
**Thesis: Computer Vision and Deep Learning for Real-Time Quality Inspection in Manufacturing**

## Overview
Train YOLOv8 variants (nano, small, medium) on three industrial defect datasets:
- **NEU-DET**: 6 defect classes (surface defects on steel strips)
- **DAGM 2007**: Binary defect detection (1 class: `defect`)
- **KSDD2**: Binary defect detection (1 class: `defect`, heavily imbalanced)

## Pipeline
1. Load preprocessed data from Drive (output of Notebook 01)
2. Configure YOLOv8 training (hyperparameters, augmentation)
3. Train YOLOv8n, YOLOv8s, YOLOv8m on each dataset
4. Evaluate with mAP@50, mAP@50-95, Precision, Recall
5. Save best models to Drive (`models/yolo/full/`)
6. Checkpoint after each training run

## Datasets & Splits (from Notebook 01)
- Train: 70%, Val: 15%, Test: 15% (stratified)
- KSDD2: class weights applied for imbalance

## Models Trained
| Model | Params | Notes |
|-------|--------|-------|
| YOLOv8n | ~3M | Nano — fastest, least accurate |
| YOLOv8s | ~11M | Small — balanced |
| YOLOv8m | ~26M | Medium — most accurate |

In [ ]:
# ---- Fix for PyTorch 2.6+ + Ultralytics: force torch.load(weights_only=False) ----
import torch

_original_torch_load = torch.load

def _torch_load_weights_only_false(*args, **kwargs):
    # Force old behavior used by many libs pre-2.6
    kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)

torch.load = _torch_load_weights_only_false

print("✅ Patched torch.load to use weights_only=False")


✅ Patched torch.load to use weights_only=False


In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────────
# Install required packages
!pip install ultralytics>=8.0.0 pandas openpyxl tqdm -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set seeds for reproducibility
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    print(f'PyTorch version: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'CUDA device: {torch.cuda.get_device_name(0)}')
except ImportError:
    print('PyTorch not found')

# Define Drive root
DRIVE_ROOT = '/content/drive/MyDrive/thesis_project'
print(f'\nDrive root: {DRIVE_ROOT}')

# Print GPU info
!nvidia-smi

Mounted at /content/drive
PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA device: Tesla T4

Drive root: /content/drive/MyDrive/thesis_project
Thu Feb 19 13:04:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       3MiB /  15360MiB |      0%      D

In [ ]:
import pathlib

DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/thesis_project")
DAGM_DIR = DRIVE_ROOT / "datasets" / "DAGM"

(DAGM_DIR).mkdir(parents=True, exist_ok=True)

yaml_path = DAGM_DIR / "data.yaml"
yaml_content = f"""path: {DAGM_DIR}
train: images/train
val: images/val
test: images/test
nc: 1
names: ['defect']
"""

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("Wrote:", yaml_path)
print("Exists?", yaml_path.exists())


Wrote: /content/drive/MyDrive/thesis_project/datasets/DAGM/data.yaml
Exists? True


In [ ]:
# ── Cell 2: Verify Notebook 01 Outputs ───────────────────────────────────────
import json
import pathlib

# Load notebook01 status
status_file = pathlib.Path(DRIVE_ROOT) / 'checkpoints/notebook01_status.json'
if not status_file.exists():
    raise FileNotFoundError(
        f'Notebook 01 checkpoint not found at {status_file}.\n'
        'Please run Notebook 01 (data preprocessing) first.'
    )

with open(status_file, 'r') as f:
    nb01_status = json.load(f)

print('Notebook 01 status loaded:')
print(json.dumps(nb01_status, indent=2))

# Verify each dataset's data.yaml exists
DATASETS_CHECK = ['NEU-DET', 'DAGM', 'KSDD2']
missing = []

print('\n' + '='*60)
print('Dataset verification:')
print('='*60)

for ds in DATASETS_CHECK:
    yaml_path = pathlib.Path(DRIVE_ROOT) / 'datasets' / ds / 'data.yaml'
    exists = yaml_path.exists()
    status_icon = '[OK]' if exists else '[MISSING]'
    print(f'  {status_icon}  {ds}: {yaml_path}')
    if not exists:
        missing.append(ds)

if missing:
    raise FileNotFoundError(
        f'Missing data.yaml for datasets: {missing}\n'
        'Re-run Notebook 01 to generate the preprocessed datasets.'
    )

print('\nAll dataset data.yaml files verified successfully.')
print(f'Ready to train on: {DATASETS_CHECK}')

Notebook 01 status loaded:
{
  "notebook": "01_data_download_and_preprocessing",
  "completed": true,
  "timestamp": "2026-02-18T15:07:50.013339Z",
  "seed": 42,
  "split_ratios": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "datasets": {
    "NEU-DET": {
      "nc": 6,
      "classes": [
        "crazing",
        "inclusion",
        "patches",
        "pitted_surface",
        "rolled_in_scale",
        "scratched"
      ],
      "annotation": "PASCAL VOC XML \u2192 YOLO format",
      "task": "multi-class detection + classification",
      "splits": {
        "train": 1085,
        "val": 232,
        "test": 233
      }
    },
    "DAGM": {
      "nc_detection": 1,
      "nc_classification": 2,
      "classes_used": [
        1,
        2,
        3,
        4,
        5,
        6
      ],
      "annotation": "mask PNG \u2192 cv2.findContours \u2192 YOLO format",
      "task": "binary detection + classification",
      "splits": {
        "train": 4830,
        "v

In [ ]:
# ── Cell 3: Training Hyperparameters ─────────────────────────────────────────
import os

YOLO_CONFIG = {
    'epochs': 40,
    'imgsz': 640,
    'batch': 16,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'patience': 20,
    'seed': SEED,
    'device': 0,  # GPU index
    'workers': 4,
    'project': f'{DRIVE_ROOT}/training_logs/yolo',
    'verbose': True,
    'save': True,
    'plots': True,
}

# Full model filenames (downloaded by ultralytics automatically)
MODELS = ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt']

# Corresponding base names (used for run naming and file paths)
MODEL_BASE_NAMES = ['yolov8n', 'yolov8s']

DATASETS = ['NEU-DET', 'DAGM', 'KSDD2']


# Initialize results log
results_log = []

print('YOLO Training Configuration:')
print('='*60)
for k, v in YOLO_CONFIG.items():
    print(f'  {k:20s}: {v}')
print('='*60)
print(f'Models   : {MODELS}')
print(f'Datasets : {DATASETS}')


YOLO Training Configuration:
  epochs              : 40
  imgsz               : 640
  batch               : 16
  optimizer           : AdamW
  lr0                 : 0.001
  lrf                 : 0.01
  momentum            : 0.937
  weight_decay        : 0.0005
  warmup_epochs       : 3
  patience            : 20
  seed                : 42
  device              : 0
  workers             : 4
  project             : /content/drive/MyDrive/thesis_project/training_logs/yolo
  verbose             : True
  save                : True
  plots               : True
Models   : ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt']
Datasets : ['NEU-DET', 'DAGM', 'KSDD2']


In [ ]:
# ── Cell 4: Full Training Loop ────────────────────────────────────────────────
from ultralytics import YOLO
import shutil
import os
import json
import time
import pandas as pd
import numpy as np

# Reset results log (safe to re-run cell)
results_log = []

for dataset in DATASETS:

    data_yaml = f'{DRIVE_ROOT}/datasets/{dataset}/data.yaml'

    # Validate data.yaml exists before starting any model for this dataset
    if not os.path.exists(data_yaml):
        print(f'[ERROR] data.yaml not found for {dataset}: {data_yaml}')
        continue

    for model_name, base_name in zip(MODELS, MODEL_BASE_NAMES):
        run_name = f'{dataset}_{base_name}'
        save_path = f'{DRIVE_ROOT}/models/yolo/full/{run_name}_best.pt'

        # ── Checkpoint-resume: skip if already trained ──────────────────────
        if os.path.exists(save_path):
            print(f'[SKIP] {run_name} already trained, loading existing model')
            results_log.append({
                'dataset': dataset,
                'model': base_name,
                'status': 'skipped',
            })
            continue

        print(f'\n{"="*60}')
        print(f'Training {run_name}')
        print(f'{"="*60}')
        t0 = time.time()

        # Load pretrained YOLOv8 weights
        model = YOLO(model_name)

        # Build training args from shared config
        train_args = {**YOLO_CONFIG}
        train_args['data'] = data_yaml
        train_args['name'] = run_name
        train_args['exist_ok'] = True

        # For single-class datasets, nc is already defined correctly in data.yaml
        # single_cls=False preserves per-class stats even with 1 class
        if dataset in ['DAGM', 'KSDD2']:
            train_args['single_cls'] = False

        # ── Run training ────────────────────────────────────────────────────
        results = model.train(**train_args)

        elapsed = time.time() - t0

        # ── Copy best weights to persistent Drive location ──────────────────
        best_weights = f'{YOLO_CONFIG["project"]}/{run_name}/weights/best.pt'
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        shutil.copy(best_weights, save_path)
        print(f'Saved best model to {save_path}')

        # ── Record metrics ───────────────────────────────────────────────────
        metrics = results.box
        entry = {
            'dataset': dataset,
            'model': base_name,
            'map50': round(float(metrics.map50), 4),
            'map50_95': round(float(metrics.map), 4),
            'precision': round(float(metrics.mp), 4),
            'recall': round(float(metrics.mr), 4),
            'train_time_min': round(elapsed / 60, 1),
            'status': 'trained',
        }
        results_log.append(entry)
        print(f'Results: {entry}')

        # ── Intermediate save of results log ────────────────────────────────
        os.makedirs(f'{DRIVE_ROOT}/results/tables', exist_ok=True)
        pd.DataFrame(results_log).to_csv(
            f'{DRIVE_ROOT}/results/tables/yolo_train_results_partial.csv',
            index=False
        )

print('\nAll training complete!')
print('='*60)
df_results = pd.DataFrame(results_log)
print(df_results.to_string())

# Save final training summary
os.makedirs(f'{DRIVE_ROOT}/results/tables', exist_ok=True)
df_results.to_csv(
    f'{DRIVE_ROOT}/results/tables/yolo_train_results.csv',
    index=False
)
print(f'\nTraining summary saved to: {DRIVE_ROOT}/results/tables/yolo_train_results.csv')


[SKIP] NEU-DET_yolov8n already trained, loading existing model
[SKIP] NEU-DET_yolov8s already trained, loading existing model
[SKIP] DAGM_yolov8n already trained, loading existing model
[SKIP] DAGM_yolov8s already trained, loading existing model
[SKIP] KSDD2_yolov8n already trained, loading existing model
[SKIP] KSDD2_yolov8s already trained, loading existing model

All training complete!
   dataset    model   status
0  NEU-DET  yolov8n  skipped
1  NEU-DET  yolov8s  skipped
2     DAGM  yolov8n  skipped
3     DAGM  yolov8s  skipped
4    KSDD2  yolov8n  skipped
5    KSDD2  yolov8s  skipped

Training summary saved to: /content/drive/MyDrive/thesis_project/results/tables/yolo_train_results.csv


In [ ]:
# ── Cell 5: Test-set Evaluation ───────────────────────────────────────────────
from ultralytics import YOLO
import pandas as pd
import os

test_results = []

for dataset in DATASETS:
    data_yaml = f'{DRIVE_ROOT}/datasets/{dataset}/data.yaml'

    for base_name in MODEL_BASE_NAMES:
        run_name = f'{dataset}_{base_name}'
        model_path = f'{DRIVE_ROOT}/models/yolo/full/{run_name}_best.pt'

        if not os.path.exists(model_path):
            print(f'[SKIP] Model not found: {model_path}')
            continue

        print(f'Evaluating {run_name} on test split...')
        model = YOLO(model_path)

        # Run validation on the held-out test split
        metrics = model.val(
            data=data_yaml,
            split='test',
            device=0,
            imgsz=640,
            batch=16,
            verbose=False,
            project=f'{DRIVE_ROOT}/training_logs/yolo',
            name=f'{run_name}_test_eval',
            exist_ok=True,
        )

        mp = float(metrics.box.mp)
        mr = float(metrics.box.mr)
        f1 = 2 * mp * mr / (mp + mr + 1e-9)

        entry = {
            'dataset': dataset,
            'model': base_name,
            'split': 'test',
            'map50': round(float(metrics.box.map50), 4),
            'map50_95': round(float(metrics.box.map), 4),
            'precision': round(mp, 4),
            'recall': round(mr, 4),
            'f1': round(f1, 4),
        }
        test_results.append(entry)
        print(f'  {entry}')

# Save test results to CSV
df_test = pd.DataFrame(test_results)
os.makedirs(f'{DRIVE_ROOT}/results/tables', exist_ok=True)
csv_path = f'{DRIVE_ROOT}/results/tables/yolo_test_results.csv'
df_test.to_csv(csv_path, index=False)

print('\nTest results saved to:', csv_path)
print('='*60)
print(df_test.to_string())

Evaluating NEU-DET_yolov8n on test split...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 80.4±178.3 ms, read: 0.0±0.0 MB/s, size: 15.4 KB)
val: Scanning /content/drive/MyDrive/thesis_project/datasets/NEU-DET/labels/test... 233 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 233/233 1.5it/s 2:33
val: /content/drive/MyDrive/thesis_project/datasets/NEU-DET/images/test/patches_198.jpg: 1 duplicate labels removed
val: New cache created: /content/drive/MyDrive/thesis_project/datasets/NEU-DET/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.3it/s 6.7s
                   all        233        571      0.634      0.719      0.724        0.4
Speed: 2.4ms preprocess, 5.2ms inference, 0.0ms loss, 4.9ms postprocess per image
Results saved to /content/drive/MyDrive/

In [ ]:
# ── Cell 6: Training Curves (from Ultralytics results.csv) ───────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import os

figs_dir = f'{DRIVE_ROOT}/results/figures'
os.makedirs(figs_dir, exist_ok=True)

for dataset in DATASETS:
    for base_name in MODEL_BASE_NAMES:
        run_name = f'{dataset}_{base_name}'
        results_csv = f'{DRIVE_ROOT}/training_logs/yolo/{run_name}/results.csv'

        if not os.path.exists(results_csv):
            print(f'[SKIP] No results.csv for {run_name}')
            continue

        df = pd.read_csv(results_csv)
        # Strip whitespace from column names (Ultralytics sometimes adds spaces)
        df.columns = df.columns.str.strip()

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(f'Training Curves: {run_name}', fontsize=14, fontweight='bold')

        # ── Box loss ─────────────────────────────────────────────────────────
        ax = axes[0]
        if 'train/box_loss' in df.columns and 'val/box_loss' in df.columns:
            ax.plot(df['epoch'], df['train/box_loss'], label='train', color='steelblue')
            ax.plot(df['epoch'], df['val/box_loss'], label='val', color='coral')
        ax.set_title('Box Loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # ── mAP@50 ───────────────────────────────────────────────────────────
        ax = axes[1]
        if 'metrics/mAP50(B)' in df.columns:
            ax.plot(df['epoch'], df['metrics/mAP50(B)'], color='green')
        ax.set_title('mAP@50')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('mAP@50')
        ax.grid(True, alpha=0.3)

        # ── mAP@50-95 ────────────────────────────────────────────────────────
        ax = axes[2]
        if 'metrics/mAP50-95(B)' in df.columns:
            ax.plot(df['epoch'], df['metrics/mAP50-95(B)'], color='purple')
        ax.set_title('mAP@50-95')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('mAP@50-95')
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        save_path = f'{figs_dir}/FIG_YOLO_Training_{run_name}.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Saved: {save_path}')

print('\nTraining curve plots complete.')

Saved: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_Training_NEU-DET_yolov8n.png
Saved: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_Training_NEU-DET_yolov8s.png
Saved: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_Training_DAGM_yolov8n.png
Saved: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_Training_DAGM_yolov8s.png
Saved: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_Training_KSDD2_yolov8n.png
Saved: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_Training_KSDD2_yolov8s.png

Training curve plots complete.


In [ ]:
# ── Cell 7: Confusion Matrices ────────────────────────────────────────────────
# Ultralytics saves confusion matrix PNGs during val — copy them to results/figures/
import shutil
import os
import glob

figs_dir = f'{DRIVE_ROOT}/results/figures'
os.makedirs(figs_dir, exist_ok=True)
eval_log_dir = f'{DRIVE_ROOT}/training_logs/yolo'

copied = 0
skipped = 0

for dataset in DATASETS:
    for base_name in MODEL_BASE_NAMES:
        run_name = f'{dataset}_{base_name}'
        eval_dir = f'{eval_log_dir}/{run_name}_test_eval'

        if not os.path.exists(eval_dir):
            print(f'[SKIP] Eval dir not found: {eval_dir}')
            skipped += 1
            continue

        # Find confusion matrix PNGs (Ultralytics names them confusion_matrix*.png)
        cm_files = glob.glob(f'{eval_dir}/*.png')
        found = False
        for f in cm_files:
            basename = os.path.basename(f).lower()
            if 'confusion' in basename:
                dest = f'{figs_dir}/FIG_YOLO_CM_{run_name}.png'
                shutil.copy(f, dest)
                print(f'Copied confusion matrix: {dest}')
                copied += 1
                found = True

        if not found:
            print(f'[WARN] No confusion matrix PNG found in: {eval_dir}')
            skipped += 1

print(f'\nConfusion matrix collection complete. Copied: {copied}, Skipped/Missing: {skipped}')

Copied confusion matrix: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_CM_NEU-DET_yolov8n.png
Copied confusion matrix: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_CM_NEU-DET_yolov8n.png
Copied confusion matrix: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_CM_NEU-DET_yolov8s.png
Copied confusion matrix: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_CM_NEU-DET_yolov8s.png
Copied confusion matrix: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_CM_DAGM_yolov8n.png
Copied confusion matrix: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_CM_DAGM_yolov8n.png
Copied confusion matrix: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_CM_DAGM_yolov8s.png
Copied confusion matrix: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_CM_DAGM_yolov8s.png
Copied confusion matrix: /content/drive/MyDrive/thesis_project/results/figures/FIG_YOLO_CM_KSDD2_yolov8n.png
Copied confusio

In [ ]:
# ── Cell 8: Checkpoint ────────────────────────────────────────────────────────
import json
import os
import datetime

checkpoint = {
    'notebook': '02_yolo_detection_training',
    'completed': True,
    'timestamp': datetime.datetime.now().isoformat(),
    'models_trained': results_log,
    'test_results_csv': f'{DRIVE_ROOT}/results/tables/yolo_test_results.csv',
    'figures_dir': f'{DRIVE_ROOT}/results/figures/',
    'models_dir': f'{DRIVE_ROOT}/models/yolo/full/',
}

os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
ckpt_path = f'{DRIVE_ROOT}/checkpoints/notebook02_status.json'
with open(ckpt_path, 'w') as f:
    json.dump(checkpoint, f, indent=2)

print('='*60)
print('Notebook 02 Complete!')
print(f'Checkpoint saved to: {ckpt_path}')
print(f'Models saved to: {DRIVE_ROOT}/models/yolo/full/')
print(f'Results CSV: {DRIVE_ROOT}/results/tables/yolo_test_results.csv')
print('='*60)
print('\nCheckpoint contents:')
print(json.dumps(checkpoint, indent=2))

Notebook 02 Complete!
Checkpoint saved to: /content/drive/MyDrive/thesis_project/checkpoints/notebook02_status.json
Models saved to: /content/drive/MyDrive/thesis_project/models/yolo/full/
Results CSV: /content/drive/MyDrive/thesis_project/results/tables/yolo_test_results.csv

Checkpoint contents:
{
  "notebook": "02_yolo_detection_training",
  "completed": true,
  "timestamp": "2026-02-19T13:17:03.150179",
  "models_trained": [
    {
      "dataset": "NEU-DET",
      "model": "yolov8n",
      "status": "skipped"
    },
    {
      "dataset": "NEU-DET",
      "model": "yolov8s",
      "status": "skipped"
    },
    {
      "dataset": "DAGM",
      "model": "yolov8n",
      "status": "skipped"
    },
    {
      "dataset": "DAGM",
      "model": "yolov8s",
      "status": "skipped"
    },
    {
      "dataset": "KSDD2",
      "model": "yolov8n",
      "status": "skipped"
    },
    {
      "dataset": "KSDD2",
      "model": "yolov8s",
      "status": "skipped"
    }
  ],
  "test_results